# Notebook 1 - Demonstracja klasy `Chatbot`

W tym notebooku pokazujemy uzycie klasy `Chatbot` bezposrednio z poziomu Pythona
(bez warstwy REST API). Pokazujemy:

1. inicjalizacje bota i prompt systemowy,
2. wymiane wiadomosci (z historia),
3. dzialanie kontroli dlugosci kontekstu,
4. obsluge bledow.

Aby uruchomic z prawdziwym modelem, najpierw uruchom Ollame:
```bash
ollama serve
ollama pull llama3.2
```

Jezeli Ollama nie jest dostepna, notebook automatycznie przelaczy sie na
podstawiony klient (mock), zeby wszystkie komorki dalo sie wykonac.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from app.config import settings
from app.chatbot import Chatbot
from app.llm_client import LLMClient


def make_bot(system_prompt=None):
    """Probuje uzyc prawdziwego klienta; w razie braku Ollamy - mock."""
    try:
        client = LLMClient(settings)
        client.generate([{"role": "user", "content": "ping"}])
        print("Uzywam prawdziwego modelu:", settings.llm_model)
        return Chatbot(cfg=settings, llm_client=client, system_prompt=system_prompt)
    except Exception as exc:
        print("Ollama niedostepna ({}). Uzywam fake klienta.".format(type(exc).__name__))

        class FakeLLM:
            def generate(self, messages):
                last_user = next((m for m in reversed(messages) if m["role"] == "user"), {"content": ""})
                return f"[fake] Twoja wiadomosc to: {last_user['content']}"

        return Chatbot(cfg=settings, llm_client=FakeLLM(), system_prompt=system_prompt)


bot = make_bot()

2026-05-23 08:38:20 | INFO     | chatbot | Chatbot zainicjalizowany | model=llama3.2 | max_ctx=3000 tok | max_hist=20 msg


Ollama niedostepna (APIConnectionError). Uzywam fake klienta.


## 1. Pierwsza rozmowa

Wywolujemy `bot.chat(...)`. Zwraca obiekt `ChatResult` z polami:
- `reply` - tekst odpowiedzi modelu,
- `ok` - czy wywolanie sie powiodlo,
- `used_tokens` - liczba tokenow uzytych w tym requeste,
- `history_length` - liczba wiadomosci w historii.

In [2]:
r1 = bot.chat("Czesc! Kim jestes?")
print("Odpowiedz:", r1.reply)
print("Meta:", {"ok": r1.ok, "tokens": r1.used_tokens, "history": r1.history_length})

2026-05-23 08:38:20 | INFO     | chatbot | USER: Czesc! Kim jestes?


2026-05-23 08:38:20 | INFO     | chatbot | ASSISTANT: [fake] Twoja wiadomosc to: Czesc! Kim jestes?


Odpowiedz: [fake] Twoja wiadomosc to: Czesc! Kim jestes?
Meta: {'ok': True, 'tokens': 103, 'history': 2}


## 2. Historia rozmowy - bot pamieta kontekst

W drugim zapytaniu odwolujemy sie do poprzedniej wypowiedzi. Model widzi cala
historie (zapisana w `bot.memory.history`), wiec moze rozumiec kontekst.

In [3]:
r2 = bot.chat("Powtorz moje poprzednie pytanie slowo w slowo.")
print(r2.reply)
print("\nHistoria po dwoch turach:")
for m in bot.history:
    print(f"  [{m['role']:>9}]: {m['content'][:80]}")

2026-05-23 08:38:20 | INFO     | chatbot | USER: Powtorz moje poprzednie pytanie slowo w slowo.


2026-05-23 08:38:20 | INFO     | chatbot | ASSISTANT: [fake] Twoja wiadomosc to: Powtorz moje poprzednie pytanie slowo w slowo.


[fake] Twoja wiadomosc to: Powtorz moje poprzednie pytanie slowo w slowo.

Historia po dwoch turach:
  [     user]: Czesc! Kim jestes?
  [assistant]: [fake] Twoja wiadomosc to: Czesc! Kim jestes?
  [     user]: Powtorz moje poprzednie pytanie slowo w slowo.
  [assistant]: [fake] Twoja wiadomosc to: Powtorz moje poprzednie pytanie slowo w slowo.


## 3. Wlasny prompt systemowy

Mozemy nadac botowi rolne i osobowosc poprzez `system_prompt`.

In [4]:
poeta = make_bot(system_prompt="Jestes poeta. Odpowiadasz tylko czterowierszami.")
print(poeta.chat("Powiedz cos o Krakowie.").reply)

2026-05-23 08:38:21 | INFO     | chatbot | Chatbot zainicjalizowany | model=llama3.2 | max_ctx=3000 tok | max_hist=20 msg


2026-05-23 08:38:21 | INFO     | chatbot | USER: Powiedz cos o Krakowie.


2026-05-23 08:38:21 | INFO     | chatbot | ASSISTANT: [fake] Twoja wiadomosc to: Powiedz cos o Krakowie.


Ollama niedostepna (APIConnectionError). Uzywam fake klienta.
[fake] Twoja wiadomosc to: Powiedz cos o Krakowie.


## 4. Kontrola dlugosci kontekstu (sliding window)

Tworzymy bota z bardzo malym oknem kontekstu i wysylamy duzo dlugich wiadomosci.
Historia rosnie nieskonczenie, ale do modelu trafiaja tylko te najnowsze
(mieszczace sie w `max_context_tokens`).

In [5]:
small_cfg = settings.model_copy(update={"max_context_tokens": 250, "max_history_messages": 50})

class EchoLLM:
    def generate(self, messages):
        return "OK"

small_bot = Chatbot(cfg=small_cfg, llm_client=EchoLLM(), system_prompt="Jestes botem.")

for i in range(10):
    r = small_bot.chat(f"Wiadomosc {i} " + ("powtorzona dluga tresc " * 8))

msgs, used = small_bot.memory.build_messages()
print(f"Pelna historia ma {len(small_bot.history)} wiadomosci")
print(f"Do modelu wysylamy tylko {len(msgs)} ostatnich wiadomosci ({used} tokenow w limicie 250)")

2026-05-23 08:38:21 | INFO     | chatbot | Chatbot zainicjalizowany | model=llama3.2 | max_ctx=250 tok | max_hist=50 msg


2026-05-23 08:38:21 | INFO     | chatbot | USER: Wiadomosc 0 powtorzona dluga tresc powtorzona dluga tresc powtorzona dluga tresc powtorzona dluga tresc powtorzona dluga...


2026-05-23 08:38:21 | INFO     | chatbot | ASSISTANT: OK


2026-05-23 08:38:21 | INFO     | chatbot | USER: Wiadomosc 1 powtorzona dluga tresc powtorzona dluga tresc powtorzona dluga tresc powtorzona dluga tresc powtorzona dluga...


2026-05-23 08:38:21 | INFO     | chatbot | ASSISTANT: OK


...

2026-05-23 08:38:21 | INFO     | chatbot | USER: Wiadomosc 8 powtorzona dluga tresc powtorzona dluga tresc powtorzona dluga tresc powtorzona dluga tresc powtorzona dluga...


2026-05-23 08:38:21 | INFO     | chatbot | ASSISTANT: OK


2026-05-23 08:38:21 | INFO     | chatbot | USER: Wiadomosc 9 powtorzona dluga tresc powtorzona dluga tresc powtorzona dluga tresc powtorzona dluga tresc powtorzona dluga...


2026-05-23 08:38:21 | INFO     | chatbot | ASSISTANT: OK


Pelna historia ma 20 wiadomosci
Do modelu wysylamy tylko 7 ostatnich wiadomosci (247 tokenow w limicie 250)


## 5. Obsluga bledow

Podstawiamy klient ktory zglasza blad polaczenia z API.
Bot ladnie loguje blad, zwraca komunikat dla uzytkownika i sprawia ze historia
zostaje spojna (cofa wiadomosc bez odpowiedzi).

In [6]:
import httpx
from openai import APIConnectionError

class BrokenLLM:
    def generate(self, messages):
        raise APIConnectionError(request=httpx.Request("POST", "http://localhost:11434"))

broken_bot = Chatbot(cfg=settings, llm_client=BrokenLLM())
r = broken_bot.chat("test")
print("ok:", r.ok)
print("error:", r.error)
print("reply:", r.reply)
print("historia po bledzie (pusta - cofamy):", broken_bot.history)

2026-05-23 08:38:21 | INFO     | chatbot | Chatbot zainicjalizowany | model=llama3.2 | max_ctx=3000 tok | max_hist=20 msg


2026-05-23 08:38:21 | INFO     | chatbot | USER: test


2026-05-23 08:38:21 | ERROR    | chatbot | Brak polaczenia z serwerem LLM (czy Ollama dziala?)


ok: False
error: connection_error
reply: Nie moge polaczyc sie z modelem. Sprawdz czy serwer LLM dziala.
historia po bledzie (pusta - cofamy): []


## 6. Reset rozmowy

In [7]:
print("Przed resetem:", len(bot.history), "wiadomosci")
bot.reset()
print("Po resecie:", len(bot.history), "wiadomosci")

2026-05-23 08:38:21 | INFO     | chatbot | Historia rozmowy wyczyszczona


Przed resetem: 4 wiadomosci
Po resecie: 0 wiadomosci
